In [1]:
import pandas as pd

data = pd.read_csv("OneSearchBib_data.csv")
print("Records:", len(data))
print("Columns:", list(data.columns))

Records: 5000
Columns: ['Title', 'Vernacular Title', 'Creator', 'Contributor', 'Is Part Of', 'Dissertation', 'Subject', 'MESH subjects', 'Genre', 'Description', 'Contents', 'Other title', 'Related titles', 'Series', 'Publisher', 'Date', 'Edition', 'Format', 'Frequency', 'Source', 'Bound with', 'permalink']


In [2]:
for column in ["Title", "Description", "Subject", "Date"]:
    blanks = data[column].fillna("").astype(str).str.strip().eq("").sum()
    print(column, "blank:", blanks)

Title blank: 0
Description blank: 312
Subject blank: 285
Date blank: 5000


In [3]:
data[["Date", "Is Part Of"]].head(5)

,Date,Is Part Of
0,NaN,"Sustainability, 2024-10, Vol.16 (19), p.8520, ..."
1,NaN,"International journal on e-learning, 2023, Vol..."
2,NaN,"Educational research review, 2022-02, Vol.35, ..."
3,NaN,"Frontiers in medicine, 2026-04, Vol.13, p.1787..."
4,NaN,"Annals of medicine (Helsinki), 2024-12, Vol.56..."


In [4]:
titles = data["Title"].fillna("").str.strip().str.casefold()
print("Records with a title seen earlier:", titles.duplicated().sum())

Records with a title seen earlier: 91


In [5]:
possible_duplicates = data.loc[
    titles.duplicated(keep=False), ["Title", "Is Part Of"]
].copy()

print(possible_duplicates.sort_values("Title").head(12).to_string(index=False, max_colwidth=80))

                                                                           Title                                                                       Is Part Of
15. Utilization of Augmented Reality Smart Glasses for the Provision of Remot... Plastic and reconstructive surgery. Global open, 2022-06, Vol.10 (5S Suppl 1)...
15. Utilization of Augmented Reality Smart Glasses for the Provision of Remot...     Plastic and reconstructive surgery. Global open, 2022-06, Vol.10 (5S), p.7-8
         A Mobile Augmented Reality system for the learning of dental morphology                                Digital education review, 2016-12 (30), p.234-247
         A Mobile Augmented Reality system for the learning of dental morphology                                     Digital education review, 2017-01, p.234-247
A Novel Scenario-Based, Mixed-Reality Platform for Training Nontechnical Skil...                 JMIR serious games, 2022-12, Vol.10 (4), p.e40727, Article 40727
A Novel Scenario-Based, Mixe

In [6]:
print("Completely identical rows:", data.duplicated().sum())

Completely identical rows: 0


In [7]:
for i in data["Description"].dropna().sample(5, random_state=42).index:
    print("\nRow", i + 1, ":", data.at[i, "Description"][:400])


Row 1397 : There is a widespread belief that VR technologies can provide controlled, multi-sensory, interactive 3D stimulus environments that engage patients in interventions and measure, record and motivate required human performance. In order to investigate state-of-the-art and associated occupations we provided a careful review of 6 leading medical and technical bibliometric databases. Despite the apparen

Row 3416 : In robotic surgery, surgical planning and surgical navigation represent two crucial elements, allowing surgeons to maximize surgical outcomes while minimizing the risk of complications. In this context, an emerging imaging technology, namely augmented reality (AR), can represent a powerful tool to create an integration of preoperative 3D models into the live intraoperative view, providing an inter

Row 847 : Virtual Reality (VR) simulation is increasingly being adopted as a potential revolutionary technology in surgical education, providing the ability to offer an imme

In [8]:
lengths = data["Description"].fillna("").str.len()
print(lengths.describe(percentiles=[0.10, 0.50, 0.90, 0.99]).round())

count     5000.0
mean      1516.0
std        741.0
min          0.0
10%        652.0
50%       1520.0
90%       2326.0
99%       3452.0
max      10000.0
Name: Description, dtype: float64


In [9]:
i = lengths.idxmax()
print("Title:", data.at[i, "Title"])
print("Beginning:", data.at[i, "Description"][:500])
print("End:", data.at[i, "Description"][-300:])

Title: Academic Librarians' Attitudes towards Augmented Reality
Beginning: Background.Augmented reality (AR) is a digital layer in the physical environment, which adds virtual elements to the real world. Academic libraries must meet certain criteria to enter fourth-generation libraries, one of which is the use of new technologies. The aim of this study was to investigate academic librarians' views on the application of components of AR technology in the libraries of medical universities. Methods. The study was a cross-sectional survey study. The statistical population 
End:  the existence of appropriate content, usefulness, providing diverse services, user guidance, location and opportunity to implement augmented reality on a larger scale. The application of AR technology, measured by 12 items from the point of view of librarians, had the highest score compared to othe


In [10]:
print("Descriptions exactly 10,000 characters:", (lengths == 10000).sum())

Descriptions exactly 10,000 characters: 1


In [11]:
print("Descriptions with 1–49 characters:", lengths.between(1, 49).sum())

Descriptions with 1–49 characters: 5


In [12]:
short = data.loc[lengths.between(1, 49), ["Title", "Description"]]
print(short.to_string(index=False, max_colwidth=100))

                                                                                               Title                                Description
Training in lung cancer surgery through the metaverse, including extended reality, in the smart o...                   na KCI Citation Count: 3
The digital metaverse: Applications in artificial intelligence, medical education, and integrativ...                     KCI Citation Count: 43
The expansion of virtual learning and virtual reality in nursing education: Opportunities and cha... No abstract is available for this article.
Application of computer-based testing in the Korean Medical Licensing Examination, the emergence ...                      KCI Citation Count: 4
Educational innovation in virtual and augmented reality for health professional education: a Sout...                      KCI Citation Count: 0


In [13]:
working = data.copy()
working["usable_description"] = lengths >= 50

print(working["usable_description"].value_counts())

usable_description
True     4683
False     317
Name: count, dtype: int64


In [14]:
working.insert(0, "source_row", range(2, len(working) + 2))
print(working[["source_row", "Title", "usable_description"]].head())

   source_row                                              Title  \
0           2  Effectiveness of Virtual Reality in Healthcare...   
1           3  Nursing education through the lens of immersiv...   
2           4  A systematic review of immersive technology ap...   
3           5  The use of the extended reality technologies i...   
4           6  Metaverse-based simulation: a scoping review o...   

   usable_description  
0                True  
1                True  
2                True  
3                True  
4                True  


In [15]:
working["repeated_title"] = titles.duplicated(keep=False)
print(working["repeated_title"].value_counts())

repeated_title
False    4825
True      175
Name: count, dtype: int64


In [16]:
working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")
print("Saved", len(working), "records")

Saved 5000 records


In [17]:
columns = ["source_row", "Title", "Is Part Of", "Description", "permalink"]
working.loc[working["repeated_title"], columns].sort_values("Title").to_csv(
    "repeated_titles_review.csv", index=False, encoding="utf-8-sig"
)
print("Saved repeated-title records:", working["repeated_title"].sum())

Saved repeated-title records: 175


In [18]:
print(working["Genre"].fillna("[blank]").value_counts().head(15).to_string())

Genre
[blank]    5000


In [19]:
publication_text = (
    working["Title"].fillna("") + " " +
    working["Is Part Of"].fillna("") + " " +
    working["Dissertation"].fillna("")
)

for term in ["thesis", "dissertation", "conference", "proceedings", "editorial"]:
    count = publication_text.str.contains(
        rf"\b{term}\b", case=False, regex=True
    ).sum()
    print(term, count)

thesis 0
dissertation 0
conference 29
proceedings 34
editorial 10


In [20]:
pattern = r"\b(thesis|dissertation|conference|proceedings|editorial)\b"
flagged = working[
    publication_text.str.contains(pattern, case=False, regex=True)
]

print(
    flagged[["source_row", "Title", "Is Part Of"]]
    .head(15)
    .to_string(index=False, max_colwidth=90)
)

 source_row                                                                                      Title                                                                                 Is Part Of
         21 Editorial: VR, AR, MR in healthcare: the role of immersive technologies in medical trai...                                     Frontiers in digital health, 2025-08, Vol.7, p.1669899
        108 C-CASE 2024: Surgical Education Through Innovation01. A 25-year retrospective of Canadi...                           Canadian journal of surgery, 2025-02, Vol.68 (1suppl1), p.S1-S13
        140 Editorial – Beyond the Mannikin: Charting Research Trajectories for Nursing Simulation’...                   Journal of interactive learning research, 2023-07, Vol.34 (3), p.429-441
        220 Editorial: Future prospects of learning in the clinical environment: exploring the tech...                         Frontiers in medicine, 2025-09, Vol.12, p.1673779, Article 1673779
        242 C-CASE 2022: Compe

C:\Users\ffahmee\AppData\Local\Temp\ipykernel_5668\1538951103.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  publication_text.str.contains(pattern, case=False, regex=True)


In [21]:
working["editorial_title"] = (
    working["Title"].fillna("").str.match(r"(?i)^\s*editorial\b")
)
print("Titles beginning Editorial:", working["editorial_title"].sum())

Titles beginning Editorial: 10


In [22]:
source = working["Is Part Of"].fillna("")
mask = source.str.contains(r"\b(?:conference|proceedings)\b", case=False, regex=True)

print(
    source[mask].str.split(",").str[0]
    .value_counts()
    .head(20)
    .to_string()
)

Is Part Of
AIP conference proceedings                                                                                                                                                    11
Proceedings of the International Symposium of Human Factors and Ergonomics in Healthcare                                                                                       7
Proceedings of the Human Factors and Ergonomics Society Annual Meeting                                                                                                         5
Journal of physics. Conference series                                                                                                                                          3
2025 47th Annual International Conference of the IEEE Engineering in Medicine and Biology Society (EMBC)                                                                       2
AMIA ... Annual Symposium proceedings                                                                   

In [23]:
conference_pattern = (
    r"AIP conference proceedings|"
    r"International Symposium of Human Factors and Ergonomics in Healthcare|"
    r"Human Factors and Ergonomics Society Annual Meeting|"
    r"Journal of physics\. Conference series|"
    r"International Conference of the IEEE Engineering in Medicine|"
    r"AMIA .*Annual Symposium proceedings|"
    r"Spring Simulation Conference|"
    r"Engineering proceedings"
)

working["conference_source"] = working["Is Part Of"].fillna("").str.contains(
    conference_pattern, case=False, regex=True
)

print("Clear conference-source records:", working["conference_source"].sum())

Clear conference-source records: 36


In [24]:
print(
    "Records with Dissertation information:",
    working["Dissertation"].fillna("").astype(str).str.strip().ne("").sum()
)

Records with Dissertation information: 0


In [25]:
blank_publication = (
    working["Is Part Of"].fillna("").astype(str).str.strip().eq("")
)
print("Blank Is Part Of:", blank_publication.sum())

Blank Is Part Of: 0


In [26]:
print(working["Format"].fillna("[blank]").value_counts().head(15).to_string())

Format
[blank]    5000


In [27]:
working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")
print("Saved", len(working), "records")

Saved 5000 records


In [28]:
details = (
    working["Is Part Of"].fillna("") + " " +
    working["Source"].fillna("")
)

thesis_clues = details.str.contains(
    r"proquest|thes|dissert|doctoral|master.s degree",
    case=False, regex=True
)

print("Records with thesis/dissertation clues:", thesis_clues.sum())
print(
    working.loc[thesis_clues, ["source_row", "Title", "Is Part Of"]]
    .head(10)
    .to_string(index=False, max_colwidth=80)
)

Records with thesis/dissertation clues: 2388
 source_row                                                                            Title                                                                Is Part Of
          2 Effectiveness of Virtual Reality in Healthcare Education: Systematic Review a...                Sustainability, 2024-10, Vol.16 (19), p.8520, Article 8520
          5 The use of the extended reality technologies in simulation-based health profe...        Frontiers in medicine, 2026-04, Vol.13, p.1787542, Article 1787542
          7 Extended reality and computer-based simulation for teaching situational aware... Advances in simulation (London), 2025-04, Vol.10 (1), p.18-31, Article 18
          9 Use of extended reality head-mounted displays in US health care education: a ...        Frontiers in medicine, 2026-04, Vol.13, p.1798546, Article 1798546
         10 Theoretical foundations and implications of augmented reality, virtual realit...  Advances in simulation (Lo

In [29]:
print(working["Source"].fillna("").head(3).str[:300].to_string(index=False))

CAB Abstracts With Full Text; Open Access资源_MDP...
ERIC; Unpaywall for CDI: Periodical Content; Al...
Elsevier ScienceDirect Journals Complete; Web o...


In [30]:
publication = working["Is Part Of"].fillna("")
thesis_label = publication.str.contains(
    r"\b(?:thesis|dissertation|doctoral)\b",
    case=False, regex=True
)

print("Explicit thesis/dissertation labels:", thesis_label.sum())
print(publication[thesis_label].head(10).to_string(index=False))

Explicit thesis/dissertation labels: 1
Journal of doctoral nursing practice, 2025-12, ...


In [31]:
working["candidate_text_analysis"] = (
    working["usable_description"]
    & ~working["editorial_title"]
    & ~working["conference_source"]
)

print(working["candidate_text_analysis"].value_counts())

candidate_text_analysis
True     4644
False     356
Name: count, dtype: int64


In [32]:
repeated = working.loc[working["repeated_title"]].copy()
repeated["normalized_title"] = titles[working["repeated_title"]]

creator_counts = repeated.groupby("normalized_title")["Creator"].nunique(dropna=False)

print("Repeated-title groups:", len(creator_counts))
print("Groups with different Creator entries:", (creator_counts > 1).sum())

Repeated-title groups: 84
Groups with different Creator entries: 35


In [33]:
examples = creator_counts[creator_counts > 1].index[:5]

print(
    repeated.loc[
        repeated["normalized_title"].isin(examples),
        ["source_row", "Title", "Creator", "Is Part Of"]
    ]
    .sort_values("Title")
    .to_string(index=False, max_colwidth=70)
)

 source_row                                                                  Title                                                                Creator                                                 Is Part Of
       2826 A Mobile Augmented Reality system for the learning of dental morpho... Juan, M Carmen ; Alexandrescu, Lucian ; Folguera, Fernando ; García...          Digital education review, 2016-12 (30), p.234-247
       3160 A Mobile Augmented Reality system for the learning of dental morpho... Juan, M. Carmen ; Alexandrescu, Lucian ; Folguera, Fernando ; Garcí...               Digital education review, 2017-01, p.234-247
        343 Analysis of research trends in the healthcare field utilizing exten...                   Ji-eun Im ; Ju-hee Lee ; Soon-ryun Lim ; Won-jae Lee                       한국치위생학회지, 2024-06, Vol.24 (3), p.197
       3970 Analysis of research trends in the healthcare field utilizing exten...               Im, Ji-Eun ; Lee, Ju-Hee ; Lim, Soon-Ryun ; Lee, Wo

In [34]:
description_counts = repeated.groupby("normalized_title")["Description"].nunique(
    dropna=False
)

print("Groups with identical descriptions:", (description_counts == 1).sum())
print("Groups with different descriptions:", (description_counts > 1).sum())

Groups with identical descriptions: 33
Groups with different descriptions: 51


In [1]:
print(len(working))

NameError: name 'working' is not defined

In [2]:
import pandas as pd

working = pd.read_csv("bibliography_working.csv")
print("Reloaded records:", len(working))
print("Saved flags:", [c for c in working.columns if c not in pd.read_csv("OneSearchBib_data.csv", nrows=0).columns])

Reloaded records: 5000
Saved flags: ['source_row', 'usable_description', 'repeated_title', 'editorial_title', 'conference_source']


In [8]:
normalized_title = working["Title"].fillna("").str.strip().str.casefold()
description = working["Description"].fillna("").str.strip()

working["duplicate_text_record"] = pd.DataFrame({
    "title": normalized_title,
    "description": description
}).duplicated(keep="first")

print("Extra records with matching title and description:",
      working["duplicate_text_record"].sum())

Extra records with matching title and description: 35


In [9]:
repeated = working.loc[working["repeated_title"]].copy()
repeated["normalized_title"] = normalized_title[working["repeated_title"]]
repeated["description_length"] = repeated["Description"].fillna("").str.len()

counts = repeated.groupby("normalized_title")["Description"].nunique(dropna=False)
examples = counts[counts > 1].index[:3]

print(
    repeated.loc[
        repeated["normalized_title"].isin(examples),
        ["source_row", "Title", "Is Part Of", "description_length"]
    ]
    .sort_values("Title")
    .to_string(index=False, max_colwidth=65)
)

 source_row                                                             Title                                                       Is Part Of  description_length
        776 A Novel Scenario-Based, Mixed-Reality Platform for Training No... JMIR serious games, 2022-12, Vol.10 (4), p.e40727, Article 40727                1937
       2527 A Novel Scenario-Based, Mixed-Reality Platform for Training No...           JMIR serious games, 2022-10, Vol.10 (4), Article 40727                1937
        765 A mixed-reality stimulator for lumbar puncture training: a pil... BMC medical education, 2023-03, Vol.23 (1), p.178-6, Article 178                1607
       2639 A mixed-reality stimulator for lumbar puncture training: a pil...          BMC medical education, 2023-03, Vol.23 (1), Article 178                1600
        606 AI Virtual Humans Empowering Nursing Education: Opening a New ...                      Hu li za zhi, 2026-02, Vol.73 (1), p.e26103                1024
       3181 AI Virtual

In [10]:
import re

repeated["normalized_description"] = repeated["Description"].fillna("").map(
    lambda text: re.sub(r"\W+", "", text.casefold())
)

normalized_counts = repeated.groupby("normalized_title")[
    "normalized_description"
].nunique()

print("Repeated-title groups identical after spacing/punctuation cleanup:",
      (normalized_counts == 1).sum())
print("Groups still different:", (normalized_counts > 1).sum())

Repeated-title groups identical after spacing/punctuation cleanup: 46
Groups still different: 38


In [11]:
repeated["publication_name"] = (
    repeated["Is Part Of"].fillna("")
    .str.split(",").str[0]
    .str.strip()
    .str.casefold()
)

different_groups = normalized_counts[normalized_counts > 1].index
journal_counts = repeated[
    repeated["normalized_title"].isin(different_groups)
].groupby("normalized_title")["publication_name"].nunique()

print("Different-text groups in one publication:", (journal_counts == 1).sum())
print("Different-text groups across publications:", (journal_counts > 1).sum())

Different-text groups in one publication: 35
Different-text groups across publications: 3


In [12]:
across_publications = journal_counts[journal_counts > 1].index

print(
    repeated.loc[
        repeated["normalized_title"].isin(across_publications),
        ["source_row", "Title", "Is Part Of", "description_length"]
    ]
    .sort_values("Title")
    .to_string(index=False, max_colwidth=80)
)

 source_row                                                                            Title                                                                       Is Part Of  description_length
       1833                                               DIGITALIZATION IN DENTAL EDUCATION       International journal of medical dentistry, 2025-04, Vol.29 (2), p.160-160                   0
       2186                                               Digitalization in dental education Revista de la Facultad de Odontología Universidad de Antioquia, 2023-02, Vol....                1065
       4387 Gynaecological Endoscopic Surgical Education and Assessment. A diploma progra... European Journal of Obstetrics & Gynecology and Reproductive Biology, 2016-04...                1629
       4746 Gynaecological endoscopic surgical education and assessment. A diploma progra...                            Gynecological surgery, 2016-08, Vol.13 (3), p.133-137                1629
       1581                   

In [13]:
normalized_description_all = working["Description"].fillna("").map(
    lambda text: re.sub(r"\W+", "", text.casefold())
)

working["duplicate_model_text"] = pd.DataFrame({
    "title": normalized_title,
    "description": normalized_description_all
}).duplicated(keep="first")

print("Extra matching-text records:", working["duplicate_model_text"].sum())

Extra matching-text records: 50


In [14]:
working["candidate_text_analysis"] = (
    working["usable_description"]
    & ~working["editorial_title"]
    & ~working["conference_source"]
    & ~working["duplicate_model_text"]
)

print("Provisional modeling records:", working["candidate_text_analysis"].sum())

Provisional modeling records: 4604


In [15]:
sample = working.loc[
    working["candidate_text_analysis"],
    ["source_row", "Title"]
].sample(15, random_state=26)

print(sample.to_string(index=False, max_colwidth=110))

 source_row                                                                                                          Title
       3688 Metaverse-Integrated Scenario-Based Professional Training Based on the Social Learning Theory: Evidence Fro...
        198                                                    Augmented reality in surgical training: a systematic review
       3662 Distance-Based Educational Interventions for Empathy and Compassionate Care: A Systematic Review of Interna...
       2952                                          The Year in Graduate Medical Education: Selected Highlights from 2024
       2825 The Effectiveness of a Question-Embedded Movie Clips Learning Program in Nursing Students: A Quasi-Experime...
       3611 Pulse!!: A Model for Research and Development of Virtual-Reality Learning in Military Medical Education and...
       4802 Evidence of Skill Transferability From Training in Virtual Reality Robotic Surgery Simulators to Nonvirtual...
       1191     

In [16]:
text = working["Title"].fillna("") + " " + working["Description"].fillna("")
xr_pattern = (
    r"virtual reality|augmented reality|mixed reality|extended reality|"
    r"\bVR\b|\bAR\b|\bMR\b|\bXR\b|metaverse|immersive"
)

working["mentions_xr"] = text.str.contains(xr_pattern, case=False, regex=True)

print(
    "Provisional records with no XR term:",
    (working["candidate_text_analysis"] & ~working["mentions_xr"]).sum()
)

Provisional records with no XR term: 722


In [17]:
no_xr = working.loc[
    working["candidate_text_analysis"] & ~working["mentions_xr"],
    ["source_row", "Title"]
].sample(12, random_state=27)

print(no_xr.to_string(index=False, max_colwidth=110))

 source_row                                                                                                          Title
       3568            Artificial Intelligence in Psychological Practice: Current and Future Applications and Implications
       4379                                    3D-printed pediatric endoscopic ear surgery simulator for surgical training
       2733         Use of two virtual dissection tables in a postgraduate oral anatomy course: feedback from the students
       2209                    Role of Visuohaptic Surgical Training Simulator in Resident Education of Orthopedic Surgery
       3441         Addressing the Impact of COVID-19 on the Residency Application Process Through a Virtual Subinternship
       2847 Innovative nursing training in prehospital trauma care: design and impact assessment using a dual-cabin amb...
       2299               Innovations in Nursing Education at the Intersection of a Changing Learner and a Changing System
       2793 The 

In [18]:
other_pattern = (
    r"haptic|virtual (?:environment|simulator|simulation|world|dissection|anatomy)|"
    r"3D (?:simulation|simulator|environment)|head.mounted"
)

working["other_immersive_clue"] = text.str.contains(
    other_pattern, case=False, regex=True
)

print(
    "No XR term, but another immersive clue:",
    (working["candidate_text_analysis"]
     & ~working["mentions_xr"]
     & working["other_immersive_clue"]).sum()
)

No XR term, but another immersive clue: 168


In [19]:
no_technology_clue = (
    working["candidate_text_analysis"]
    & ~working["mentions_xr"]
    & ~working["other_immersive_clue"]
)

print(
    working.loc[no_technology_clue, ["source_row", "Title"]]
    .sample(15, random_state=28)
    .to_string(index=False, max_colwidth=110)
)

 source_row                                                                                                          Title
       2701                                                                       Künstliche Intelligenz in der Ausbildung
        617 Exploring Applications of Artificial Intelligence Tools in Clinical Care and Health Professions Education: ...
       1647                                                Dental education in Singapore: A journey of 90 years and beyond
       2894 Restructuring Physical Therapy Education After COVID-19: A Narrative Review on the Global Perspectives and ...
       2066 Introduction, availability and role of simulation in surgical education and training: Review of current evi...
       2363                                                                                        開拓臨床實務與知能的新途徑－遊戲式護理教學應用
       1637 Research landscape thematic evolution and future trends of artificial intelligence in medical education fro...
       2985     

In [20]:
no_technology_clue = (
    working["candidate_text_analysis"]
    & ~working["mentions_xr"]
    & ~working["other_immersive_clue"]
)

print(
    working.loc[no_technology_clue, ["source_row", "Title"]]
    .sample(15, random_state=28)
    .to_string(index=False, max_colwidth=110)
)

 source_row                                                                                                          Title
       2701                                                                       Künstliche Intelligenz in der Ausbildung
        617 Exploring Applications of Artificial Intelligence Tools in Clinical Care and Health Professions Education: ...
       1647                                                Dental education in Singapore: A journey of 90 years and beyond
       2894 Restructuring Physical Therapy Education After COVID-19: A Narrative Review on the Global Perspectives and ...
       2066 Introduction, availability and role of simulation in surgical education and training: Review of current evi...
       2363                                                                                        開拓臨床實務與知能的新途徑－遊戲式護理教學應用
       1637 Research landscape thematic evolution and future trends of artificial intelligence in medical education fro...
       2985     

In [21]:
working["needs_scope_review"] = no_technology_clue

print("Needs scope review:", working["needs_scope_review"].sum())
print("Provisional records without this flag:",
      (working["candidate_text_analysis"] & ~working["needs_scope_review"]).sum())

Needs scope review: 554
Provisional records without this flag: 4050


In [22]:
working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")
print("Saved", len(working), "records")

Saved 5000 records


In [23]:
first_pass = working[
    working["candidate_text_analysis"] & ~working["needs_scope_review"]
]

print(first_pass[["source_row", "Title"]]
      .sample(12, random_state=42)
      .to_string(index=False, max_colwidth=100))

 source_row                                                                                                Title
       3405       New Approach to Accelerated Image Annotation by Leveraging Virtual Reality and Cloud Computing
        161                                              Extended reality in medical education: a scoping review
       2396 Merged virtual reality teaching of the fundamentals of laparoscopic surgery: a randomized control...
       2873                      Acquisition of Fire Safety Knowledge and Skills With Virtual Reality Simulation
       3877 CVRriculum Program: Outcomes from an Exploratory Pilot Program Incorporating Virtual Reality into...
       2088     Enhancement of Anatomical Education Using Augmented Reality: An Empirical Study of Body Painting
       3079 Participating experience of virtual reality teaching among nursing students: a meta-synthesis of ...
       2389 Training on a virtual reality laparoscopic simulator improves performance of live la

In [24]:
for _, row in working[working["source_row"].isin([3405, 2873, 4133])].iterrows():
    print("\nROW:", row["source_row"])
    print("TITLE:", row["Title"])
    print("DESCRIPTION:", str(row["Description"])[:700])


ROW: 2873
TITLE: Acquisition of Fire Safety Knowledge and Skills With Virtual Reality Simulation
DESCRIPTION: BACKGROUNDPrelicensure nursing students seeking to enter perioperative nursing need preparatory fire safety knowledge and skills training to participate as a member of an operating room (OR) team.
PURPOSEThis pilot study examined the effectiveness of the Virtual Electrosurgery Skill Trainer (VEST) on OR fire safety skills among prelicensure nursing students.
METHODSAn experimental pretest-posttest design was used in this study. Twenty nursing students were randomized to a control or an intervention group. Knowledge and skills acquisition of OR fire safety were assessed.
RESULTSThere were no statistically significant findings in knowledge for either group. Fisher exact test demonstrated signi

ROW: 3405
TITLE: New Approach to Accelerated Image Annotation by Leveraging Virtual Reality and Cloud Computing
DESCRIPTION: Three-dimensional imaging is at the core of medical imaging an

In [25]:
working["manual_scope_review"] = working["source_row"].isin([3405, 4133])
print(working.loc[working["manual_scope_review"], ["source_row", "Title"]].to_string(index=False))

 source_row                                                                                                        Title
       3405               New Approach to Accelerated Image Annotation by Leveraging Virtual Reality and Cloud Computing
       4133 THE ROLE OF VIRTUAL REALITY THERAPY IN DENTISTRY: A BIBLIOMETRIC ANALYSIS OF CURRENT TRENDS AND APPLICATIONS


In [26]:
learning_pattern = (
    r"educat|train|teach|learn|student|curricul|skill|competenc|"
    r"resident|instruction|pedagog|simulat"
)

working["learning_clue"] = (
    working["Title"].fillna("") + " " + working["Description"].fillna("")
).str.contains(learning_pattern, case=False, regex=True)

check = working[
    working["candidate_text_analysis"]
    & ~working["needs_scope_review"]
    & ~working["learning_clue"]
]

print("Records without a learning clue:", len(check))
print(check[["source_row", "Title"]]
      .sample(min(12, len(check)), random_state=42)
      .to_string(index=False, max_colwidth=100))

Records without a learning clue: 52
 source_row                                                                                                Title
       1471                                         Chapter 2. The Digital Meets the Physical and the Biological
       3728 Augmented reality with intraoperative indocyanine green lymphatic mapping in colorectal cancer: p...
       4605 Virtual Reality for Anxiety Reduction: A Mixed-Methods Preliminary Study on Blue Space Environmen...
       1047     Potenzial der Digitalisierung in Aus‑, Fort- und Weiterbildung in Orthopädie und Unfallchirurgie
       3834     Encountered-Type Haptic Interface for Representation of Shape and Rigidity of 3D Virtual Objects
        468 A Bibliometric Analysis of Technology in Digital Health: Exploring Health Metaverse and Visualizi...
       1262 Exploring the Potential of Three-Dimensional Imaging, Printing, and Modeling in Pediatric Surgica...
       4834 Mixed reality enhances interobserver reliability

In [27]:
learning_pattern = (
    r"educat|train|teach|learn|student|curricul|skill|competenc|"
    r"resident|instruction|pedagog|simulat"
)

working["learning_clue"] = (
    working["Title"].fillna("") + " " + working["Description"].fillna("")
).str.contains(learning_pattern, case=False, regex=True)

check = working[
    working["candidate_text_analysis"]
    & ~working["needs_scope_review"]
    & ~working["learning_clue"]
]

print("Records without a learning clue:", len(check))
print(check[["source_row", "Title"]]
      .sample(min(12, len(check)), random_state=42)
      .to_string(index=False, max_colwidth=100))

Records without a learning clue: 52
 source_row                                                                                                Title
       1471                                         Chapter 2. The Digital Meets the Physical and the Biological
       3728 Augmented reality with intraoperative indocyanine green lymphatic mapping in colorectal cancer: p...
       4605 Virtual Reality for Anxiety Reduction: A Mixed-Methods Preliminary Study on Blue Space Environmen...
       1047     Potenzial der Digitalisierung in Aus‑, Fort- und Weiterbildung in Orthopädie und Unfallchirurgie
       3834     Encountered-Type Haptic Interface for Representation of Shape and Rigidity of 3D Virtual Objects
        468 A Bibliometric Analysis of Technology in Digital Health: Exploring Health Metaverse and Visualizi...
       1262 Exploring the Potential of Three-Dimensional Imaging, Printing, and Modeling in Pediatric Surgica...
       4834 Mixed reality enhances interobserver reliability

In [28]:
print(check[["source_row", "Title"]]
      .sort_values("source_row")
      .to_string(index=False, max_colwidth=140))

 source_row                                                                                                                                        Title
         52                                                                                                                         虛擬實境與擴增實境在醫護實務與教育之應用
         84      Usability Study of Augmented Reality Visualization Modalities on Localization Accuracy in the Head and Neck: Randomized Crossover Trial
        214                                  Parents’ Perspectives on the Use of Virtual Reality in Pediatric Chronic Pain Management: Qualitative Study
        270 Patient Perception of the Use of the Apple Vision Pro Virtual Reality Headset in an Anesthesiology Informed Consent Process: Prospective ...
        306 Effectiveness of Virtual Reality Interventions on Perioperative Anxiety, Depression, Blood Pressure, and Heart Rate: Systematic Review an...
        468    A Bibliometric Analysis of Technology in Digital Health: Exploring 

In [29]:
working["needs_learning_review"] = working["source_row"].isin(check["source_row"])
working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")
print("Saved. Records needing learning-scope review:",
      working["needs_learning_review"].sum())

Saved. Records needing learning-scope review: 52


In [30]:
title_learning = working["Title"].fillna("").str.contains(
    learning_pattern, case=False, regex=True
)

review = working[
    working["candidate_text_analysis"]
    & ~working["needs_scope_review"]
    & working["learning_clue"]
    & ~title_learning
]

print("Learning clue in description but not title:", len(review))
print(review[["source_row", "Title"]]
      .sample(min(12, len(review)), random_state=42)
      .to_string(index=False, max_colwidth=110))

Learning clue in description but not title: 621
 source_row                                                                                                          Title
        262 Current Applications and Limitations of Augmented Reality in Urological Surgery: A Practical Primer and ‘St...
       4704                                Virtual interactive suturing for the Fundamentals of Laparoscopic Surgery (FLS)
        411                       The application of virtual reality and augmented reality in Oral & Maxillofacial Surgery
       2219 Artificial Intelligence and Augmented Reality in Orthopedic Surgery: A Narrative Review of Current Applicat...
        562                                   Applications of mixed reality technology in neurosurgery: a narrative review
       4915                              Visualization of Heart Electrocardiographic Activity using a Mixed Reality System
       1763                    Augmented Reality in Neurosurgery, State of Art and Future P

In [31]:
for _, row in working[working["source_row"].isin([262, 4704, 2513, 4915])].iterrows():
    print("\nROW:", row["source_row"])
    print("TITLE:", row["Title"])
    print("DESCRIPTION:", str(row["Description"])[:900])


ROW: 262
TITLE: Current Applications and Limitations of Augmented Reality in Urological Surgery: A Practical Primer and ‘State of the Field’
DESCRIPTION: Purpose of Review
To provide a primer for how augmented reality (AR)-guided surgical technology works at a fundamental level and discuss recent advances and limitations in a rapidly advancing field, including studies aiming to reduce current issues limiting wider adoption.
Recent Findings
Among the studies published within the last five years, AR-guided technologies have advanced from pre-operative planning to intraoperative use in procedures including robot-assisted radical prostatectomy, percutaneous nephrolithotomy, and renal transplantation. Artificial intelligence (AI) and deep learning techniques have allowed for development of automatic registration to address challenges with soft tissue deformation. Subspecialities which may benefit from further AR/MR adoption include reconstructive and andrology, which were underrepresented 

In [32]:
working.loc[
    working["source_row"].isin([262, 2513]),
    "manual_scope_review"
] = True

print(working.loc[working["manual_scope_review"],
                  ["source_row", "Title"]].to_string(index=False))

 source_row                                                                                                                        Title
        262 Current Applications and Limitations of Augmented Reality in Urological Surgery: A Practical Primer and ‘State of the Field’
       2513        Using Augmented Reality as a Clinical Support Tool to Assist Combat Medics in the Treatment of Tension Pneumothoraces
       3405                               New Approach to Accelerated Image Annotation by Leveraging Virtual Reality and Cloud Computing
       4133                 THE ROLE OF VIRTUAL REALITY THERAPY IN DENTISTRY: A BIBLIOMETRIC ANALYSIS OF CURRENT TRENDS AND APPLICATIONS


In [33]:
machine_learning = review["Description"].fillna("").str.contains(
    r"machine learning|deep learning", case=False, regex=True
)

print("Of the 621, descriptions mentioning machine/deep learning:",
      machine_learning.sum())
print(review.loc[machine_learning, ["source_row", "Title"]]
      .sample(min(10, machine_learning.sum()), random_state=42)
      .to_string(index=False, max_colwidth=110))

Of the 621, descriptions mentioning machine/deep learning: 28
 source_row                                                                                                          Title
       2472     Utilizing a multilayer perceptron artificial neural network to assess a virtual reality surgical procedure
       4612                              Artificial Intelligence in Urologic Robotic Oncologic Surgery: A Narrative Review
       2219 Artificial Intelligence and Augmented Reality in Orthopedic Surgery: A Narrative Review of Current Applicat...
       3745                 Transformative Frontiers: A Comprehensive Review of Emerging Technologies in Modern Healthcare
         83                               Metaverse and Virtual Health Care in Ophthalmology: Opportunities and Challenges
       2983 Optimized virtual reality design through user immersion level detection with novel feature fusion and expla...
       3514 ARTIFICIAL INTELLIGENCE, VIRTUAL REALITY AND ROBOTICS IN COLORECT

In [34]:
strong_learning_pattern = (
    r"educat|train|teach|student|curricul|skill|competenc|"
    r"resident|instruction|pedagog|simulat"
)

other_learning = review["Description"].fillna("").str.contains(
    strong_learning_pattern, case=False, regex=True
)

print("Without another education/training term:", (~other_learning).sum())
print(review.loc[~other_learning, ["source_row", "Title"]]
      .sample(min(12, (~other_learning).sum()), random_state=42)
      .to_string(index=False, max_colwidth=110))

Without another education/training term: 3
 source_row                                                                                                          Title
       1216 A Medical Image Visualization Technique Assisted with AI-Based Haptic Feedback for Robotic Surgery and Heal...
       3200                   Testing Communication Strategies to Convey Genomic Concepts Using Virtual Reality Technology
       3613                                 Development of SMILE-MOM: a metaverse-based support program for single mothers


In [35]:
working.loc[
    working["source_row"].isin([1216, 3200, 3613]),
    "manual_scope_review"
] = True

working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")
print("Saved. Individually flagged records:",
      working["manual_scope_review"].sum())

Saved. Individually flagged records: 7


In [36]:
clear_title = working["Title"].fillna("").str.contains(
    r"educat|train|teach|learn|student|curricul|skill|competenc|"
    r"resident|instruction|pedagog|simulat",
    case=False, regex=True
)

clear_group = working[
    working["candidate_text_analysis"]
    & ~working["needs_scope_review"]
    & clear_title
    & ~working["manual_scope_review"]
]

print("Clear-title group:", len(clear_group))
print(clear_group[["source_row", "Title"]]
      .sample(12, random_state=19)
      .to_string(index=False, max_colwidth=110))

Clear-title group: 3377
 source_row                                                                                                          Title
       1757 A Conversational Ethics Bot (CALEB) Versus GPT-4o for Medical Ethics Education: Transcript Analysis and Ext...
        354 The feasibility and usability of mixed reality teaching in a hospital setting based on self-reported percep...
       3943                                             Preparing for the Next Generation NCLEX Using a Virtual Simulation
        701 Generative Artificial Intelligence and Immersive Technology for Medical Education: Opportunities and Challe...
       3053 Impacts of Real and Virtual Environments on Construction Safety Knowledge Learning in Virtual Reality Class...
       2898                                 Competency Assessment of New Registered Nurses Using Immersive Virtual Reality
       4922                                                Integration of virtual reality into veterinary science t

In [37]:
for _, row in working[working["source_row"].isin([1757, 3053, 3138, 1205])].iterrows():
    print("\nROW:", row["source_row"])
    print("TITLE:", row["Title"])
    print("DESCRIPTION:", str(row["Description"])[:750])


ROW: 1205
TITLE: Preparing medical first responders for crises: a systematic literature review of disaster training programs and their effectiveness
DESCRIPTION: Background
Adequate training and preparation of medical first responders (MFRs) are essential for an optimal performance in highly demanding situations like disasters (e.g., mass accidents, natural catastrophes). The training needs to be as effective as possible, because precise and effective behavior of MFRs under stress is central for ensuring patients’ survival and recovery. This systematic review offers an overview of scientifically evaluated training methods used to prepare MFRs for disasters. It identifies different effectiveness indicators and provides an additional analysis of how and to what extent the innovative training technologies virtual (VR) and mixed reality (MR) are included in disaster training research.
Methods
The system

ROW: 1757
TITLE: A Conversational Ethics Bot (CALEB) Versus GPT-4o for Medical Ethics

In [38]:
working.loc[working["source_row"] == 3053, "manual_scope_review"] = True

health_pattern = (
    r"medic|health|nurs|surg|dental|dentist|pharma|clinical|"
    r"physician|patient|anatom|hospital|therap|paramedic|"
    r"radiolog|veterinar|first responder"
)

health_clue = (
    working["Title"].fillna("") + " " + working["Description"].fillna("")
).str.contains(health_pattern, case=False, regex=True)

check_health = working[
    working["candidate_text_analysis"]
    & ~working["needs_scope_review"]
    & ~working["manual_scope_review"]
    & ~health_clue
]

print("Provisional records without a health clue:", len(check_health))
print(check_health[["source_row", "Title"]]
      .sample(min(12, len(check_health)), random_state=42)
      .to_string(index=False, max_colwidth=110))

Provisional records without a health clue: 96
 source_row                                                                                                          Title
       4274                                  A Study on Effective Usage and Evaluation of the Projection Hands-Free Manual
       4192                                                      Virtual Reality Stroop Task for neurocognitive assessment
       4008 Face, Content, and Construct Validity of the Take-Home EoSim Augmented Reality Laparoscopy Simulator for Ba...
       4954                                                                               Interactive multi-modal suturing
       2321                                   Laparoscopic virtual reality and box trainers: is one superior to the other?
       4272 The impact of constructive feedback on training in gastrointestinal endoscopy using high-fidelity virtual-r...
       3834               Encountered-Type Haptic Interface for Representation of Shape and R

In [39]:
health_pattern = (
    r"medic|health|nurs|surg|dental|dentist|pharma|clinical|"
    r"physician|patient|anatom|hospital|therap|paramedic|"
    r"radiolog|veterinar|first responder|laparoscop|endoscop|"
    r"sutur|arthroscop|anesthes|anaesthes"
)

health_clue = (
    working["Title"].fillna("") + " " + working["Description"].fillna("")
).str.contains(health_pattern, case=False, regex=True)

check_health = working[
    working["candidate_text_analysis"]
    & ~working["needs_scope_review"]
    & ~working["manual_scope_review"]
    & ~health_clue
]

print("Still without a health clue:", len(check_health))
print(check_health[["source_row", "Title"]]
      .sample(min(12, len(check_health)), random_state=42)
      .to_string(index=False, max_colwidth=110))

Still without a health clue: 80
 source_row                                                                                                          Title
       2158 Past, present, and future of virtual reality-based biological science education (2010–2025): a BERTopic-dri...
         52                                                                                           虛擬實境與擴增實境在醫護實務與教育之應用
       1755                                                 Light Field Visualization for Training and Education: A Review
       2419                                                     薬学生の臨床準備教育における体験型調剤Virtual Reality （VR）トレーニングツールの教育的有用性の検証
       1471                                                   Chapter 2. The Digital Meets the Physical and the Biological
       2045         A systematic literature review of the acceptability of the use of Metaverse in education over 16 years
        494 Do small- and medium-sized companies intend to use the Metaverse as part of their strategy? A b

In [40]:
working["needs_health_review"] = working["source_row"].isin(
    check_health["source_row"]
)

working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")

print("Saved. Records needing health-scope review:",
      working["needs_health_review"].sum())

Saved. Records needing health-scope review: 80


In [41]:
working["provisional_core"] = (
    working["candidate_text_analysis"]
    & ~working["needs_scope_review"]
    & ~working["needs_learning_review"]
    & ~working["needs_health_review"]
    & ~working["manual_scope_review"]
)

print("Provisional core records:", working["provisional_core"].sum())
print("All records retained:", len(working))

Provisional core records: 3921
All records retained: 5000


In [42]:
sample = working[working["provisional_core"]].sample(6, random_state=27)

for _, row in sample.iterrows():
    description = str(row["Description"])
    print("\nROW:", row["source_row"])
    print("START:", description[:220])
    print("END:", description[-120:])


ROW: 4169
START: Background: Delivering empathetic care is an integral part of clinical practice and has been shown to directly impact patient outcomes. Virtual reality (VR) technology offers a resource and time-efficient method to enhan
END:  the VR experience.
Results: There was a significant increase in self-reported mean empathy scores post VR experience (p

ROW: 1281
START: This study is planned to examine the effect of a virtual reality-based operating room tour on nursing students' anxiety, stress and self-confidence levels experienced in the operating room setting.
It is important to kno
END: e operating room (7.93 SD 1.09) was found to be higher than that of the students in the control group (6.72 SD 1.20) (p 

ROW: 1215
START: Purpose/Objectives Dental education is undergoing a digital transformation, yet the adoption of immersive technologies such as haptic virtual reality (HVR) remains limited. This study aimed to map global adoption trends,
END: d sustainability-focused a

In [43]:
import html
import re

def clean_description(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

working["analysis_text"] = working["Description"].apply(clean_description)

print("Original:", repr(working.loc[working["source_row"] == 1910, "Description"].iloc[0][:180]))
print("Cleaned: ", repr(working.loc[working["source_row"] == 1910, "analysis_text"].iloc[0][:180]))

Original: '•Peer-led simulations in EIS enhance engagement, critical thinking, and decision-making.•Extended simulations offer in-depth exploration, improving confidence and practice readines'
Cleaned:  '•Peer-led simulations in EIS enhance engagement, critical thinking, and decision-making.•Extended simulations offer in-depth exploration, improving confidence and practice readines'


In [44]:
def clean_description(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = text.replace("•", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

working["analysis_text"] = working["Description"].apply(clean_description)

print(working.loc[
    working["source_row"] == 1910, "analysis_text"
].iloc[0][:180])

Peer-led simulations in EIS enhance engagement, critical thinking, and decision-making. Extended simulations offer in-depth exploration, improving confidence and practice readiness


In [45]:
working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")

working.loc[working["provisional_core"]].to_csv(
    "bibliography_provisional_core.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved working records:", len(working))
print("Saved provisional core records:", working["provisional_core"].sum())

Saved working records: 5000
Saved provisional core records: 3921


In [46]:
print(
    working.loc[working["provisional_core"], ["source_row", "Title"]]
    .sample(20, random_state=2026)
    .to_string(index=False, max_colwidth=110)
)

 source_row                                                                                                          Title
       3289 Implementation and Evaluation of a Virtual Reality-Based Cognitive Assessment and Rehabilitation Simulation...
       4000 Evaluation of Virtual Reality Technology and Online Teaching System for Medical Students in Ukraine During ...
       4576                                           Assessing the validity of VR as a training tool for medical students
       3398 Analysis of the Empathic Concern Subscale of the Emotional Response Questionnaire in a Study Evaluating the...
       3238 Effects of three-dimensional holograms on the academic performance of nursing students in a health assessme...
       3798                                                     Virtual reality as a therapeutic tool in modern healthcare
       4469 Creating Interprofessional Best Practices in Education for Care of the Transgender Community using 360-Virt...
       4305 Comp

In [47]:
for _, row in working[
    working["source_row"].isin([3289, 3798, 1940, 1722])
].iterrows():
    print("\nROW:", row["source_row"])
    print("TITLE:", row["Title"])
    print("DESCRIPTION:", str(row["Description"])[:850])


ROW: 1722
TITLE: Intraoperative fluoroscopy skills in distal radius fracture surgery: valid and reliable assessment on a novel immersive virtual reality simulator
DESCRIPTION: Background and purpose: Orthopedic trainees must be able to perform intraoperative fluoroscopy imaging to assess the surgical result after volar locking plate surgeries of distal radius fractures. Guided by Messick’s contemporary validity framework, the aim of our study was to gather evidence of validity for a test of proficiency for intraoperative imaging of a distal radius fracture using a novel immersive virtual reality simulator.Methods: 11 novices and 9 experienced surgeons employed at orthopedic departments completed 2 individual simulator sessions. At each session the participants performed 3 repetitions of an intraoperative fluoroscopic control of a distal radius fracture, consisting of 5 different fluoroscopic views. Several performance metrics were automatically recorded by the simulator and compared b

In [48]:
working.loc[
    working["source_row"] == 3798,
    "manual_scope_review"
] = True

working.loc[
    working["source_row"] == 3798,
    "provisional_core"
] = False

working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")

print("Provisional core:", working["provisional_core"].sum())
print("Individually flagged:", working["manual_scope_review"].sum())

Provisional core: 3920
Individually flagged: 9


In [49]:
therapy_title = working["Title"].fillna("").str.contains(
    r"therap|treatment|pain|rehabilitat|anxiety|depression",
    case=False, regex=True
)

education_title = working["Title"].fillna("").str.contains(
    r"educat|train|teach|learn|student|curricul|skill|competenc|"
    r"resident|instruction|pedagog|simulat",
    case=False, regex=True
)

therapy_check = working[
    working["provisional_core"]
    & therapy_title
    & ~education_title
]

print("Therapy-focused titles to inspect:", len(therapy_check))
print(therapy_check[["source_row", "Title"]]
      .sample(min(12, len(therapy_check)), random_state=42)
      .to_string(index=False, max_colwidth=110))

Therapy-focused titles to inspect: 21
 source_row                                                                                                          Title
        289 Efficacy of Various Virtual Reality Exposure Therapies for Chronic Low Back Pain: Systematic Review and Net...
       4395 THE APPLICATION OF NAVIGATION SYSTEM BASED ON THE AUGMENTED REALITY TECHNOLOGY FOR THE SURGICAL TREATMENT O...
       4112 An old approach to a novel problem: effect of combined balance therapy on virtual reality induced motion si...
        420 Hybrid System Mixed Reality and Marker-Less Motion Tracking for Sports Rehabilitation of Martial Arts Athletes
       1790                           My hands? Importance of personalised virtual hands in a neurorehabilitation scenario
       1234                      Augmented Reality as an Aid to Behavior Therapy for Anxiety Disorders: A Narrative Review
       3234                                    Technology and Mental Health: State of the Art for Ass

In [50]:
for _, row in therapy_check.sort_values("source_row").iterrows():
    print("\nROW:", row["source_row"])
    print("TITLE:", row["Title"])
    print("DESCRIPTION:", str(row["Description"])[:350])


ROW: 289
TITLE: Efficacy of Various Virtual Reality Exposure Therapies for Chronic Low Back Pain: Systematic Review and Network Meta-Analysis
DESCRIPTION: Background: Chronic low back pain (CLBP) is a major global health challenge. While nonpharmacological therapies are recommended, patient compliance is often hindered by kinesiophobia. Virtual reality (VR) offers an immersive, distraction-based approach, but the comparative effectiveness of different VR modalities remains unclear. Objective: The aim

ROW: 420
TITLE: Hybrid System Mixed Reality and Marker-Less Motion Tracking for Sports Rehabilitation of Martial Arts Athletes
DESCRIPTION: Rehabilitation is a vast field of research. Virtual and Augmented Reality represent rapidly emerging technologies that have the potential to support physicians in several medical activities, e.g., diagnosis, surgical training, and rehabilitation, and can also help sports experts analyze athlete movements and performance. In this study, we present t



In [51]:
working["needs_therapy_review"] = working["source_row"].isin(
    therapy_check["source_row"]
)

working["provisional_core"] = (
    working["provisional_core"] & ~working["needs_therapy_review"]
)

working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")

print("Therapy titles flagged:", working["needs_therapy_review"].sum())
print("Provisional core now:", working["provisional_core"].sum())

Therapy titles flagged: 21
Provisional core now: 3899


In [52]:
working.loc[working["provisional_core"]].to_csv(
    "bibliography_provisional_core.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Updated core file:",
      pd.read_csv("bibliography_provisional_core.csv").shape[0],
      "records")

Updated core file: 3899 records


In [53]:
flags = [
    "usable_description", "duplicate_model_text", "editorial_title",
    "conference_source", "needs_scope_review", "needs_learning_review",
    "needs_health_review", "manual_scope_review",
    "needs_therapy_review", "provisional_core"
]

cleaning_summary = pd.DataFrame({
    "item": ["original_records"] + flags,
    "count": [len(working)] + [int(working[col].sum()) for col in flags]
})

cleaning_summary.to_csv(
    "cleaning_summary.csv", index=False, encoding="utf-8-sig"
)

print(cleaning_summary.to_string(index=False))

                 item  count
     original_records   5000
   usable_description   4683
 duplicate_model_text     50
      editorial_title     10
    conference_source     36
   needs_scope_review    554
needs_learning_review     52
  needs_health_review     80
  manual_scope_review      9
 needs_therapy_review     21
     provisional_core   3899


In [54]:
review_flags = [
    "needs_scope_review", "needs_learning_review",
    "needs_health_review", "manual_scope_review",
    "needs_therapy_review"
]

scope_review = working.loc[
    working[review_flags].any(axis=1),
    ["source_row", "Title", "Description"] + review_flags
]

scope_review.to_csv(
    "bibliography_scope_review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Unique records awaiting scope review:", len(scope_review))

Unique records awaiting scope review: 705


In [55]:
print(
    working.loc[working["needs_scope_review"], ["source_row", "Title"]]
    .sample(15, random_state=31)
    .to_string(index=False, max_colwidth=110)
)

 source_row                                                                                                          Title
       4249                Porcine stomach surgical simulation model for cesarean section and cervical laceration suturing
        913 Applications of Artificial Intelligence for Nonpsychomotor Skills Training in Health Professions Education:...
       3403                                            Example of digital exhibition in anthropology and medical education
       2176                                  Shader Lamps Virtual Patients: the physical manifestation of virtual patients
       3068                                    Patient Outcomes in Simulation-Based Medical Education: A Systematic Review
       2470                  Live streaming ward rounds using wearable technology to teach medical students: a pilot study
       2233                                     The Impact of Informatics on Nursing Education: A Review of the Literature
       2299     

In [56]:
hyphenated_xr = (
    working["Title"].fillna("") + " " + working["Description"].fillna("")
).str.contains(
    r"virtual[\s-]+reality|augmented[\s-]+reality|"
    r"mixed[\s-]+reality|extended[\s-]+reality",
    case=False, regex=True
)

newly_found = working[
    working["needs_scope_review"] & hyphenated_xr
]

print("Previously flagged records containing an XR term with a hyphen:",
      len(newly_found))
print(newly_found[["source_row", "Title"]]
      .sample(min(12, len(newly_found)), random_state=42)
      .to_string(index=False, max_colwidth=110))

Previously flagged records containing an XR term with a hyphen: 21
 source_row                                                                                                         Title
        549                                          Augmented-reality-assisted intraoral scanning: An andragogical study
       4543            Interactive computer-based simulator for training in blade navigation and targeting in myringotomy
       4295 Can virtual-reality simulation ensure transthoracic echocardiography skills before trainees examine patients?
        694                                                                Medical Education 4.0: A Neurology Perspective
       2415                 An augmented-reality system prototype for guiding transcranial Doppler ultrasound examination
       1752                                                      Mixed-Reality Simulation of Minimally Invasive Surgeries
       2868                                            Construct validity of th

In [57]:
xr_pattern_v2 = (
    r"virtual[\s-]+reality|augmented[\s-]+reality|"
    r"mixed[\s-]+reality|extended[\s-]+reality|"
    r"\bVR\b|\bAR\b|\bMR\b|\bXR\b|metaverse|immersive"
)

working["mentions_xr"] = (
    working["Title"].fillna("") + " " + working["Description"].fillna("")
).str.contains(xr_pattern_v2, case=False, regex=True)

working["needs_scope_review"] = (
    working["candidate_text_analysis"]
    & ~working["mentions_xr"]
    & ~working["other_immersive_clue"]
)

print("Records needing XR-scope review after correction:",
      working["needs_scope_review"].sum())

Records needing XR-scope review after correction: 533


In [58]:
base = working["candidate_text_analysis"] & ~working["needs_scope_review"]

working["needs_learning_review"] = base & ~working["learning_clue"]
working["needs_health_review"] = (
    base & ~working["manual_scope_review"] & ~health_clue
)

working["provisional_core"] = (
    base
    & ~working["needs_learning_review"]
    & ~working["needs_health_review"]
    & ~working["manual_scope_review"]
    & ~working["needs_therapy_review"]
)

print("Learning review:", working["needs_learning_review"].sum())
print("Health review:", working["needs_health_review"].sum())
print("Updated provisional core:", working["provisional_core"].sum())

Learning review: 52
Health review: 80
Updated provisional core: 3920


In [59]:
working.to_csv(
    "bibliography_working.csv", index=False, encoding="utf-8-sig"
)

working.loc[working["provisional_core"]].to_csv(
    "bibliography_provisional_core.csv",
    index=False, encoding="utf-8-sig"
)

scope_review = working.loc[
    working[review_flags].any(axis=1),
    ["source_row", "Title", "Description"] + review_flags
]
scope_review.to_csv(
    "bibliography_scope_review.csv",
    index=False, encoding="utf-8-sig"
)

cleaning_summary["count"] = (
    [len(working)] + [int(working[col].sum()) for col in flags]
)
cleaning_summary.to_csv(
    "cleaning_summary.csv", index=False, encoding="utf-8-sig"
)

print("Core:", working["provisional_core"].sum(),
      "Review:", len(scope_review))

Core: 3920 Review: 684


In [60]:
print(
    working.loc[working["needs_scope_review"], ["source_row", "Title"]]
    .sample(20, random_state=18)
    .to_string(index=False, max_colwidth=110)
)

 source_row                                                                                                          Title
       4897 Computer model of a spectrum analyzer for a virtual laboratory: development and introduction to the educati...
       3219 Dental students’ perceptions on a simulated practice using patient‐based customised typodonts during the tr...
       4386                          Three debriefing methods in virtual patient simulation: A randomized controlled trial
       1126               Reliability and Discriminant Validity of a Checklist for Surgical Scrubbing, Gowning and Gloving
       2440      Incentivising practice with take-home laparoscopic simulators in two UK Core Surgical Training programmes
       2267 RCFLA-YOLO: a deep learning-driven framework for the automated assessment of root canal filling quality in ...
       2089                             Anatomic hepatectomy planning through mobile display visualization and interaction
       3958     

In [61]:
expanded_clue = (
    r"haptic|virtual (?:environment|simulator|simulation|world|"
    r"dissection|anatomy|patient)|3D (?:simulation|simulator|"
    r"environment)|head.mounted|hololens|hologram|holograph|"
    r"realidad (?:virtual|aumentada|mixta)"
)

working["other_immersive_clue"] = (
    working["Title"].fillna("") + " " + working["Description"].fillna("")
).str.contains(expanded_clue, case=False, regex=True)

new_scope_review = (
    working["candidate_text_analysis"]
    & ~working["mentions_xr"]
    & ~working["other_immersive_clue"]
)

print("XR review group with expanded terms:", new_scope_review.sum())
print("Additional records found:",
      working["needs_scope_review"].sum() - new_scope_review.sum())

XR review group with expanded terms: 489
Additional records found: 44


In [62]:
newly_found = working[
    working["needs_scope_review"] & ~new_scope_review
]

print(newly_found[["source_row", "Title"]]
      .sample(15, random_state=42)
      .to_string(index=False, max_colwidth=110))

 source_row                                                                                                          Title
       4781                                   Virtual patient clinical placements improve student communication competence
       3483                                    The Utility of Virtual Patient Simulations for Clinical Reasoning Education
       3511                  The use of virtual patients to teach medical students history taking and communication skills
       4749                       Empathic Communication in Virtual Education for Nursing Students: I’m Sorry to Hear That
       4689 Impacto de la realidad virtual en el fortalecimiento de destrezas psicomotoras en estudiantes de Instrument...
       4909 Potenciando la competencia en reanimación cardiopulmonar: evaluación del impacto de la realidad virtual en ...
        861    Assessment of a virtual prosthetic case planning environment for dental education – A multicentric analysis
       2176     

In [63]:
working["needs_modality_review"] = (
    working["needs_scope_review"] & ~new_scope_review
)
working["needs_scope_review"] = new_scope_review

print("XR scope review:", working["needs_scope_review"].sum())
print("Modality review:", working["needs_modality_review"].sum())

XR scope review: 489
Modality review: 44


In [64]:
modality_text = (
    working["Title"].fillna("") + " " + working["Description"].fillna("")
)

explicit_modality = modality_text.str.contains(
    r"realidad (?:virtual|aumentada|mixta)|hololens|hologram|holograph",
    case=False, regex=True
)

explicit_review = working[
    working["needs_modality_review"] & explicit_modality
]

print("Explicit technology among the 44:", len(explicit_review))
print(explicit_review[["source_row", "Title"]]
      .to_string(index=False, max_colwidth=110))

Explicit technology among the 44: 13
 source_row                                                                                                          Title
        324                                             Review of Microsoft HoloLens Applications over the Past Five Years
        428 Realidad mixta en mecatrónica quirúrgica: Innovaciones y retos en la educación en bioingeniería y práctica ...
       1189 Nursing Students’ Experience of Using HoloPatient During the Coronavirus Disease 2019 Pandemic: A Qualitati...
       1203                                  Evaluating the effectiveness of learning ear anatomy using holographic models
       1358                                            Using Holograms to Enhance Learning in Health Sciences and Medicine
       2331 Inclusión de la realidad aumentada en la formación académica del estudiantado de enfermería en la Universid...
       2470                  Live streaming ward rounds using wearable technology to teach medical stu

In [65]:
review_flags = [
    "needs_scope_review", "needs_modality_review",
    "needs_learning_review", "needs_health_review",
    "manual_scope_review", "needs_therapy_review"
]

working["provisional_core"] = (
    working["candidate_text_analysis"]
    & ~working[review_flags].any(axis=1)
)

working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")

working.loc[working["provisional_core"]].to_csv(
    "bibliography_provisional_core.csv",
    index=False, encoding="utf-8-sig"
)

scope_review = working.loc[
    working[review_flags].any(axis=1),
    ["source_row", "Title", "Description"] + review_flags
]
scope_review.to_csv(
    "bibliography_scope_review.csv",
    index=False, encoding="utf-8-sig"
)

print("Provisional core:", working["provisional_core"].sum())
print("Unique records for scope review:", len(scope_review))

Provisional core: 3920
Unique records for scope review: 684


In [66]:
flags = [
    "usable_description", "duplicate_model_text", "editorial_title",
    "conference_source", "needs_scope_review", "needs_modality_review",
    "needs_learning_review", "needs_health_review",
    "manual_scope_review", "needs_therapy_review", "provisional_core"
]

cleaning_summary = pd.DataFrame({
    "item": ["original_records"] + flags,
    "count": [len(working)] + [int(working[col].sum()) for col in flags]
})

cleaning_summary.to_csv(
    "cleaning_summary.csv", index=False, encoding="utf-8-sig"
)

print("Updated cleaning_summary.csv")

Updated cleaning_summary.csv


In [67]:
print(
    working.loc[working["needs_modality_review"], ["source_row", "Title"]]
    .sort_values("source_row")
    .to_string(index=False, max_colwidth=110)
)

 source_row                                                                                                          Title
         79 The Development and Application of Virtual Patient in Clinical Training: The New Horizon for Nursing Education
        245 Virtual Patient Simulations in Health Professions Education: Systematic Review and Meta-Analysis by the Dig...
        324                                             Review of Microsoft HoloLens Applications over the Past Five Years
        428 Realidad mixta en mecatrónica quirúrgica: Innovaciones y retos en la educación en bioingeniería y práctica ...
        861    Assessment of a virtual prosthetic case planning environment for dental education – A multicentric analysis
       1189 Nursing Students’ Experience of Using HoloPatient During the Coronavirus Disease 2019 Pandemic: A Qualitati...
       1203                                  Evaluating the effectiveness of learning ear anatomy using holographic models
       1358     

In [68]:
for _, row in explicit_review.sort_values("source_row").iterrows():
    print("\nROW:", row["source_row"])
    print("TITLE:", row["Title"])
    print("DESCRIPTION:", str(row["Description"])[:450])


ROW: 324
TITLE: Review of Microsoft HoloLens Applications over the Past Five Years
DESCRIPTION: Since Microsoft HoloLens first appeared in 2016, HoloLens has been used in various industries, over the past five years. This study aims to review academic papers on the applications of HoloLens in several industries. A review was performed to summarize the results of 44 papers (dated between January 2016 and December 2020) and to outline the research trends of applying HoloLens to different industries. This study determined that HoloLens is empl

ROW: 428
TITLE: Realidad mixta en mecatrónica quirúrgica: Innovaciones y retos en la educación en bioingeniería y práctica clínica
DESCRIPTION: Resumen Introducción: La realidad mixta (RM), combinación de realidad aumentada (RA) y virtual (RV) ha emergido como una herramienta clave en la mecatrónica quirúrgica y ha mejorado la precisión y la visualización en procedimientos clínicos. Métodos: Este estudio presenta una revisión de literatura de los 

In [69]:
include_rows = [
    428, 1189, 1203, 1358, 2331, 2551,
    3427, 4019, 4689, 4909, 4921
]

working["reviewed_include"] = working["source_row"].isin(include_rows)

print("Reviewed for inclusion:", working["reviewed_include"].sum())
print(working.loc[working["reviewed_include"],
                  ["source_row", "Title"]].to_string(index=False, max_colwidth=100))

Reviewed for inclusion: 11
 source_row                                                                                                Title
        428 Realidad mixta en mecatrónica quirúrgica: Innovaciones y retos en la educación en bioingeniería y...
       1189 Nursing Students’ Experience of Using HoloPatient During the Coronavirus Disease 2019 Pandemic: A...
       1203                        Evaluating the effectiveness of learning ear anatomy using holographic models
       1358                                  Using Holograms to Enhance Learning in Health Sciences and Medicine
       2331 Inclusión de la realidad aumentada en la formación académica del estudiantado de enfermería en la...
       2551 Assessment of classification accuracy in endodontics by dental students: a comparative study of h...
       3427 ABASI (Aprendizaje Basado en Simulación): un enfoque integral para el entrenamiento en crisis de ...
       4019 Educación Médica y Carga Cognitiva: Estudio de la Interac

In [70]:
working["provisional_core"] = (
    (working["candidate_text_analysis"]
     & ~working[review_flags].any(axis=1))
    | working["reviewed_include"]
)

unresolved = (
    working[review_flags].any(axis=1)
    & ~working["reviewed_include"]
)

print("Provisional core:", working["provisional_core"].sum())
print("Still awaiting scope review:", unresolved.sum())

Provisional core: 3931
Still awaiting scope review: 673


In [71]:
working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")

working.loc[working["provisional_core"]].to_csv(
    "bibliography_provisional_core.csv",
    index=False, encoding="utf-8-sig"
)

working.loc[
    unresolved,
    ["source_row", "Title", "Description"] + review_flags
].to_csv(
    "bibliography_scope_review.csv",
    index=False, encoding="utf-8-sig"
)

print("Saved core:", working["provisional_core"].sum())
print("Saved unresolved review:", unresolved.sum())

Saved core: 3931
Saved unresolved review: 673


In [72]:
remaining_modality = working[
    working["needs_modality_review"] & ~working["reviewed_include"]
]

display_clue = (
    remaining_modality["Title"].fillna("") + " "
    + remaining_modality["Description"].fillna("")
).str.contains(
    r"headset|head-mounted|three-dimensional|\b3D\b|"
    r"stereoscop|spatial computing|immersive display",
    case=False, regex=True
)

print("Remaining modality records:", len(remaining_modality))
print("With a display or 3D clue:", display_clue.sum())
print(remaining_modality.loc[
    display_clue, ["source_row", "Title"]
].to_string(index=False, max_colwidth=110))

Remaining modality records: 33
With a display or 3D clue: 3
 source_row                                                                                                          Title
       1661 A Virtual Counseling Application Using Artificial Intelligence for Communication Skills Training in Nursing...
       2176                                  Shader Lamps Virtual Patients: the physical manifestation of virtual patients
       2470                  Live streaming ward rounds using wearable technology to teach medical students: a pilot study


In [73]:
for _, row in working[
    working["source_row"].isin([1661, 2176])
].iterrows():
    print("\nROW:", row["source_row"])
    print("TITLE:", row["Title"])
    print("DESCRIPTION:", str(row["Description"])[:1000])


ROW: 1661
TITLE: A Virtual Counseling Application Using Artificial Intelligence for Communication Skills Training in Nursing Education: Development Study
DESCRIPTION: Background: The ability of nursing undergraduates to communicate effectively with health care providers, patients, and their family members is crucial to their nursing professions as these can affect patient outcomes. However, the traditional use of didactic lectures for communication skills training is ineffective, and the use of standardized patients is not time- or cost-effective. Given the abilities of virtual patients (VPs) to simulate interactive and authentic clinical scenarios in secured environments with unlimited training attempts, a virtual counseling application is an ideal platform for nursing students to hone their communication skills before their clinical postings.
Objective: The aim of this study was to develop and test the use of VPs to better prepare nursing undergraduates for communicating with real-l

In [74]:
description = working.loc[
    working["source_row"] == 1661, "Description"
].iloc[0]

for match in re.finditer(
    r"headset|head-mounted|three-dimensional|\b3D\b|"
    r"stereoscop|spatial computing|immersive display",
    description, flags=re.IGNORECASE
):
    print(description[max(0, match.start()-120):match.end()+120])

tbot was trained using a natural language processing engine, Google Cloud's Dialogflow, and was later visualized into a three-dimensional (3D) avatar form using Unity 3D.
Results: The VPs included four case scenarios that were congruent with the nursing und
ing a natural language processing engine, Google Cloud's Dialogflow, and was later visualized into a three-dimensional (3D) avatar form using Unity 3D.
Results: The VPs included four case scenarios that were congruent with the nursing undergr
ssing engine, Google Cloud's Dialogflow, and was later visualized into a three-dimensional (3D) avatar form using Unity 3D.
Results: The VPs included four case scenarios that were congruent with the nursing undergraduates' semesters' learning


In [75]:
working.loc[working["source_row"] == 2176, "reviewed_include"] = True

working["provisional_core"] = (
    (working["candidate_text_analysis"]
     & ~working[review_flags].any(axis=1))
    | working["reviewed_include"]
)

unresolved = (
    working[review_flags].any(axis=1)
    & ~working["reviewed_include"]
)

working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")

working.loc[working["provisional_core"]].to_csv(
    "bibliography_provisional_core.csv",
    index=False, encoding="utf-8-sig"
)

print("Provisional core:", working["provisional_core"].sum())
print("Still awaiting review:", unresolved.sum())

Provisional core: 3932
Still awaiting review: 672


In [76]:
working.loc[
    unresolved,
    ["source_row", "Title", "Description"] + review_flags
].to_csv(
    "bibliography_scope_review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Review file updated:", unresolved.sum())

Review file updated: 672


In [77]:
working["final_analysis"] = working["provisional_core"]

working.loc[working["final_analysis"]].to_csv(
    "bibliography_final_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

working.to_csv(
    "bibliography_working.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Original records:", len(working))
print("Final analysis records:", working["final_analysis"].sum())
print("Uncertain scope records:", unresolved.sum())
print("Other screening exclusions:",
      len(working) - working["final_analysis"].sum() - unresolved.sum())

Original records: 5000
Final analysis records: 3932
Uncertain scope records: 672
Other screening exclusions: 396


In [78]:
working["screening_status"] = "other_screening_exclusion"
working.loc[unresolved, "screening_status"] = "uncertain_scope"
working.loc[working["final_analysis"], "screening_status"] = "final_analysis"

final_counts = (
    working["screening_status"]
    .value_counts()
    .rename_axis("screening_status")
    .reset_index(name="records")
)

working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")
final_counts.to_csv("cleaning_summary.csv", index=False, encoding="utf-8-sig")

print(final_counts.to_string(index=False))
print("Total:", final_counts["records"].sum())

         screening_status  records
           final_analysis     3932
          uncertain_scope      672
other_screening_exclusion      396
Total: 5000


In [79]:
working["metadata_only_description"] = working["source_row"].isin(
    [1572, 3247]
)

print("Confirmed metadata-only descriptions:",
      working["metadata_only_description"].sum())

Confirmed metadata-only descriptions: 2


In [80]:
metadata_clue = working["analysis_text"].fillna("").str.contains(
    r"@|https?://|doi:|citation count|correspondence:|"
    r"no specific funding reported",
    case=False, regex=True
)

metadata_check = working[
    working["final_analysis"]
    & metadata_clue
    & ~working["metadata_only_description"]
]

print("Descriptions with metadata clues:", len(metadata_check))
print(metadata_check[["source_row", "Title"]]
      .sample(min(12, len(metadata_check)), random_state=42)
      .to_string(index=False, max_colwidth=100))

Descriptions with metadata clues: 47
 source_row                                                                                                Title
       2877 An Experimental Study on Usefulness of Virtual Reality 360° in Undergraduate Medical Education [L...
       4247 Artificial intelligence and immersive digital technologies in periodontal education: a systematic...
       2791                                              Using Mixed Reality to Practice Difficult Conversations
       4686   9 The development of a fully immersive simulation based training program for core medical trainees
       2156 Exploring the role of digital technology for feedback exchange in clinical training: a scoping re...
       3828 Transfer Validity of Pediatric Supracondylar Humeral Fracture Pin Placement Practice on In-Theate...
        821 Immersive Virtual Reality in Midwifery Education: A Narrative Review of Pedagogical Disparities, ...
       1666              Stand in surgeon’s shoes: virtual 

In [81]:
sample = metadata_check.sample(
    min(12, len(metadata_check)), random_state=42
)

for _, row in sample.iterrows():
    description = row["analysis_text"]
    print("\nROW:", row["source_row"])
    print("START:", description[:220])
    print("END:", description[-180:])


ROW: 2877
START: Peng Yao, Chloe Challen, Charlotte Caves Imperial College London, London, UKCorrespondence: Peng Yao Email peng.yao15@imperial.ac.ukWe read with considerable interest the recent article by Sultan et al1 and commend the a
END: ion the interpretation of the results of the OSCEs examination, particularly since they are not compared pre and post intervention.View the original paper by Sultan and colleagues.

ROW: 4247
START: The purpose of the systematic review was to evaluate the application and efficacy of artificial intelligence (AI) and immersive digital technologies in periodontal education. We conducted a comprehensive search of PubMed
END: r randomized controlled trials are needed to assess long-term effectiveness and cost-efficiency. https://www.crd.york.ac.uk/PROSPERO/view/CRD420251027251, PROSPERO CRD420251027251.

ROW: 2791
START: Background/Introduction: Nursing students often learn effective communication through observa-tional and varied clinical opportuniti

In [82]:
for _, row in metadata_check.sort_values("source_row").iterrows():
    print(row["source_row"], "|", row["analysis_text"][:150])

24 | This review aimed to map taxonomy frameworks, descriptions, and applications of immersive technologies in the dental literature. The Preferred reporti
45 | PurposeExtended reality, an umbrella term for virtual, augmented, and mixed reality, is increasingly used in health care education as it requires fewe
122 | Abhimanyu S Ahuja,1 Sen Yang,1 Alfredo A Paredes III,2 Daniel C Tu1,3 1Department of Ophthalmology, Casey Eye Institute, Oregon Health and Science Uni
265 | Background: Three-dimensional modalities are increasingly being used as adjuncts for medical trainees learning about complex anatomical concepts, such
375 | Background Point-of-care ultrasound (POCUS) is a diagnostic tool increasingly used. For novices, challenges such as probe orientation and hand-eye coo
613 | Over the past two decades, there have been numerous attempts at using surgical simulation software for training purposes. There has been extensive pri
647 | Working in an operating room (OR) is physically and me

In [83]:
check_rows = [122, 791, 2877, 4022, 4555, 4822]

for _, row in working[
    working["source_row"].isin(check_rows)
].iterrows():
    description = row["analysis_text"]
    print("\nROW:", row["source_row"], "LENGTH:", len(description))
    print("START:", description[:450])
    print("END:", description[-250:])


ROW: 122 LENGTH: 3576
START: Abhimanyu S Ahuja,1 Sen Yang,1 Alfredo A Paredes III,2 Daniel C Tu1,3 1Department of Ophthalmology, Casey Eye Institute, Oregon Health and Science University, Portland, OR, USA; 2Charles E. Schmidt College of Medicine, Florida Atlantic University, Boca Raton, FL, USA; 3Operative Care Division, Veterans Affairs Portland Health Care System, Portland, OR, USACorrespondence: Daniel C Tu, Department of Ophthalmology, Casey Eye Institute, Oregon Health
END: ence gaps, and assess the potential of these immersive platforms to make surgical training more affordable, efficient, and widely accessible.Keywords: virtual reality, augmented reality, cataract, retina, surgical training, graduate medical education

ROW: 791 LENGTH: 199
START: doi: https://doi.org/10.12669/pjms.40.1.8752 How to cite this: Farrukh K. Metaverse in Medical Education: A Paradigm Shift. Pak J Med Sci. 2024;40(1):255. doi: https://doi.org/10.12669/pjms.40.1.8752
END: doi: https://doi.org/10.12669

In [84]:
working["text_quality_exclude"] = working["source_row"].isin(
    [1572, 3247, 791, 4555, 2877, 4022, 4822]
)

print("Records flagged after description review:",
      working["text_quality_exclude"].sum())

Records flagged after description review: 7


In [85]:
for row_id in [122, 821]:
    description = working.loc[
        working["source_row"] == row_id, "analysis_text"
    ].iloc[0]

    print("\nROW:", row_id)
    for word in ["Abstract", "Purpose:", "Background:", "Introduction:"]:
        position = description.lower().find(word.lower())
        if position >= 0:
            print(word, "at", position,
                  "|", description[position:position + 160])


ROW: 122
Abstract at 532 | Abstract: Virtual reality (VR) and augmented reality (AR) technologies have recently gained momentum in medical education and show particular promise for ophtha

ROW: 821
Purpose: at 150 | Purpose: This narrative review synthesizes current empirical and qualitative evidence on the use of immersive Virtual Reality (VR) as an educational tool in mid


In [86]:
for row_id, marker in [(122, "Abstract:"), (821, "Purpose:")]:
    mask = working["source_row"] == row_id
    text = working.loc[mask, "analysis_text"].iloc[0]
    working.loc[mask, "analysis_text"] = text[text.find(marker):]

working["model_input"] = (
    working["final_analysis"] & ~working["text_quality_exclude"]
)

working.to_csv("bibliography_working.csv", index=False, encoding="utf-8-sig")

working.loc[working["model_input"]].to_csv(
    "bibliography_model_input.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Papers ready for analysis:", working["model_input"].sum())

Papers ready for analysis: 3925


In [87]:
for package in ["textblob", "vaderSentiment"]:
    try:
        __import__(package)
        print(package, "available")
    except ImportError:
        print(package, "not installed")

textblob not installed
vaderSentiment not installed


In [88]:
%pip install vaderSentiment

Note: you may need to restart the kernel to use updated packages.


In [89]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

model_papers["sentiment_score"] = model_papers["analysis_text"].fillna(
    ""
).apply(lambda text: analyzer.polarity_scores(text)["compound"])

print("Lowest scores:")
print(model_papers.nsmallest(5, "sentiment_score")[
    ["source_row", "Title", "sentiment_score"]
].to_string(index=False, max_colwidth=95))

print("\nHighest scores:")
print(model_papers.nlargest(5, "sentiment_score")[
    ["source_row", "Title", "sentiment_score"]
].to_string(index=False, max_colwidth=95))

NameError: name 'model_papers' is not defined

In [90]:
import pandas as pd

model_papers = pd.read_csv("topic_assignments_draft.csv")
print("Reloaded papers:", len(model_papers))

Reloaded papers: 3905


In [91]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

model_papers["sentiment_score"] = model_papers["analysis_text"].fillna(
    ""
).apply(lambda text: analyzer.polarity_scores(text)["compound"])

print("Lowest scores:")
print(model_papers.nsmallest(5, "sentiment_score")[
    ["source_row", "Title", "sentiment_score"]
].to_string(index=False, max_colwidth=95))

print("\nHighest scores:")
print(model_papers.nlargest(5, "sentiment_score")[
    ["source_row", "Title", "sentiment_score"]
].to_string(index=False, max_colwidth=95))

Lowest scores:
 source_row                                                                                           Title  sentiment_score
       1076 Development of disaster nursing education and training programs in the past 20 years (2000–2...          -0.9947
       2007        Immersive virtual reality in orthopedic surgery as elective subject for medical students          -0.9937
       2296 Comparing the Psychological Effects of Manikin-Based and Augmented Reality–Based Simulation ...          -0.9935
       3066 The effect of virtual reality based disaster training on disaster preparedness of nursing st...          -0.9901
        528 Evaluation of an immersive simulation programme for mental health clinicians to address aggr...          -0.9849

Highest scores:
 source_row                                                                                           Title  sentiment_score
       4725 Enhancing Healthcare Skills and Promoting Health Equity - A Virtual Reality-Based

In [92]:
for row_id in [2007, 3768]:
    row = model_papers.loc[
        model_papers["source_row"] == row_id
    ].iloc[0]

    print("\nROW:", row_id, "SCORE:", row["sentiment_score"])
    print("TITLE:", row["Title"])
    print("START:", row["analysis_text"][:650])
    print("END:", row["analysis_text"][-400:])


ROW: 2007 SCORE: -0.9937
TITLE: Immersive virtual reality in orthopedic surgery as elective subject for medical students
START: Background Virtual reality (VR) simulators have been introduced for skills training in various medical disciplines to create an approximately realistic environment without the risk of patient harm and have improved to more immersive VR (iVR) simulators at affordable costs. There is evidence that training on VR simulators improves technical skills but its use in orthopedic training programs and especially in curricular teaching sessions for medical students are currently not well established. The aim of this study was to describe the implementation of a VR operating theater as an elective course for undergraduate medical students and to evalua
END: te (p = 0,0087).Schlussfolgerung iVR lasst sich leicht in den Lehrplan fur Medizinstudierende integrieren und der Einsatz wird sehr gut evaluiert. iVR verbessert statistisch das Verfahrenswissen uber chirurgische Sc

In [93]:
model_papers.to_csv(
    "sentiment_scores_exploratory.csv",
    index=False,
    encoding="utf-8-sig"
)

scores = model_papers["sentiment_score"]

print("Papers scored:", len(scores))
print("Median score:", round(scores.median(), 3))
print("Positive (> 0.05):", (scores > 0.05).sum())
print("Near zero (-0.05 to 0.05):", scores.between(-0.05, 0.05).sum())
print("Negative (< -0.05):", (scores < -0.05).sum())

Papers scored: 3905
Median score: 0.927
Positive (> 0.05): 3577
Near zero (-0.05 to 0.05): 49
Negative (< -0.05): 279


In [94]:
model_papers["text_length"] = model_papers["analysis_text"].str.len()

length_groups = pd.qcut(
    model_papers["text_length"], 4,
    labels=["shortest", "short", "long", "longest"]
)

print(
    model_papers.groupby(length_groups, observed=True)["sentiment_score"]
    .agg(["count", "median"])
    .round(3)
)

             count  median
text_length               
shortest       977   0.832
short          977   0.911
long           975   0.949
longest        976   0.965


In [95]:
model_papers["tone_balance"] = model_papers["analysis_text"].fillna(
    ""
).apply(
    lambda text: (
        lambda result: result["pos"] - result["neg"]
    )(analyzer.polarity_scores(text))
)

print(
    model_papers.groupby(length_groups, observed=True)["tone_balance"]
    .agg(["count", "median"])
    .round(3)
)

             count  median
text_length               
shortest       977   0.066
short          977   0.065
long           975   0.071
longest        976   0.066


In [96]:
import json
from pathlib import Path

notebook_path = Path("bibliography_cleaning.ipynb")
notebook = json.loads(notebook_path.read_text(encoding="utf-8"))

for number, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))
    if (
        "bibliography_final_analysis.csv" in source
        or "bibliography_model_input.csv" in source
    ):
        print(number, "|", source[:140].replace("\n", " "))

105 | working["final_analysis"] = working["provisional_core"]  working.loc[working["final_analysis"]].to_csv(     "bibliography_final_analysis.csv
114 | for row_id, marker in [(122, "Abstract:"), (821, "Purpose:")]:     mask = working["source_row"] == row_id     text = working.loc[mask, "anal


In [97]:
for number in range(110, min(140, len(notebook["cells"]))):
    source = "".join(notebook["cells"][number].get("source", []))
    print(number, "|", source[:110].replace("\n", " "))

110 | for _, row in metadata_check.sort_values("source_row").iterrows():     print(row["source_row"], "|", row["anal
111 | check_rows = [122, 791, 2877, 4022, 4555, 4822]  for _, row in working[     working["source_row"].isin(check_r
112 | working["text_quality_exclude"] = working["source_row"].isin(     [1572, 3247, 791, 4555, 2877, 4022, 4822] ) 
113 | for row_id in [122, 821]:     description = working.loc[         working["source_row"] == row_id, "analysis_te
114 | for row_id, marker in [(122, "Abstract:"), (821, "Purpose:")]:     mask = working["source_row"] == row_id     
115 | for package in ["textblob", "vaderSentiment"]:     try:         __import__(package)         print(package, "av
116 | %pip install vaderSentiment
117 | from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer  analyzer = SentimentIntensityAnalyzer() 
118 | import pandas as pd  model_papers = pd.read_csv("topic_assignments_draft.csv") print("Reloaded papers:", len(m
119 | from vaderSentiment.vade